In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Import Libraries

In [2]:
pip install transformers datasets accelerate

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np

from transformers import AutoTokenizer,AutoModelForSequenceClassification, AutoModelForMultipleChoice,TrainingArguments, Trainer, EarlyStoppingCallback
from datasets import Dataset

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [4]:
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key=wandb_api_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [5]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


# EDA

In [6]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train_df.shape)
print(test_df.shape)

(2000, 8)
(500, 7)


In [7]:
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [8]:
train_df['answer'].value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

In [9]:
train_df.isnull().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

In [10]:
train_df, val_df = train_test_split(
    train_df, 
    test_size=0.2, 
    stratify=train_df["answer"], 
    random_state=4524
)

# Evaluation Metric

In [11]:
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    
    predictions = np.argsort(logits, axis=-1)[:, ::-1][:, :3]
    
    score = 0.0
    for actual, pred in zip(labels, predictions):
        if actual == pred[0]:
            score += 1.0
        elif actual == pred[1]:
            score += 0.5
        elif actual == pred[2]:
            score += 1/3
            
    return {"map3": score / len(labels)}

# Milestone 5

In [12]:
id2label = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

deberta_path = "microsoft/deberta-v3-small" 
roberta_path = "roberta-base"

deb_tok = AutoTokenizer.from_pretrained(deberta_path)
deb_model = AutoModelForSequenceClassification.from_pretrained(deberta_path, num_labels=5).eval()

rob_tok = AutoTokenizer.from_pretrained(roberta_path)
rob_model = AutoModelForSequenceClassification.from_pretrained(roberta_path, num_labels=5).eval()

def get_probs(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    with torch.no_grad():
        logits = model(**inputs).logits
    return torch.nn.functional.softmax(logits, dim=1).squeeze().numpy()

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias       

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
text_25 = str(test_df['prompt'].iloc[25])

deb_probs = get_probs(text_25, deb_model, deb_tok)
rob_probs = get_probs(text_25, rob_model, rob_tok)

# Q1
deb_top1_idx = np.argmax(deb_probs)
print(f"Q1 -> {id2label[deb_top1_idx]}, {deb_probs[deb_top1_idx]:.4f}")

# Q2
simple_avg_probs = (deb_probs + rob_probs) / 2
simple_top1_idx = np.argmax(simple_avg_probs)
print(f"Q2 -> {id2label[simple_top1_idx]}")

# Q3
weight_deb, weight_rob = 0.7, 0.3
weighted_probs = (weight_deb * deb_probs) + (weight_rob * rob_probs)
weighted_top1_idx = np.argmax(weighted_probs)
print(f"Q3 -> {id2label[weighted_top1_idx]}")

# Q4
top3_indices = np.argsort(weighted_probs)[-3:][::-1] 
top3_letters = [id2label[i] for i in top3_indices]
q4_answer = " ".join(top3_letters)
print(f"Q4 -> {q4_answer}")

Q1 -> E, 0.2418
Q2 -> E
Q3 -> E
Q4 -> E C B


In [14]:
# Q5

predictions = []

for idx, row in test_df.iterrows():
    text = str(row['prompt'])
    
    p_deb = get_probs(text, deb_model, deb_tok)
    p_rob = get_probs(text, rob_model, rob_tok)
    
    p_final = (0.7 * p_deb) + (0.3 * p_rob)
    
    top3_idx = np.argsort(p_final)[-3:][::-1]
    top3_str = " ".join([id2label[i] for i in top3_idx])
    
    predictions.append({'id': row['id'] if 'id' in row else idx, 'prediction': top3_str})

sub_df = pd.DataFrame(predictions)
sub_df.to_csv('submission.csv', index=False)

print(f"Q5 -> {len(sub_df)}")

Q5 -> 500


In [15]:
# Q6
tta_changed_count = 0

for i in range(min(50, len(test_df))):
    original_text = str(test_df['prompt'].iloc[i])
    augmented_text = "Answer the following multiple-choice question carefully: " + original_text
    
    p_orig = get_probs(original_text, deb_model, deb_tok)
    p_aug = get_probs(augmented_text, deb_model, deb_tok)
    
    p_tta_avg = (p_orig + p_aug) / 2
    
    if np.argmax(p_orig) != np.argmax(p_tta_avg):
        tta_changed_count += 1

print(f"Q6 -> {tta_changed_count}")

Q6 -> 33


In [16]:
top1_changes = 0
positive_gain_count = 0
top3_changes = 0

for i in range(min(100, len(test_df))):
    text = str(test_df['prompt'].iloc[i])
    
    p_deb = get_probs(text, deb_model, deb_tok)
    p_rob = get_probs(text, rob_model, rob_tok)
    p_ens = (0.7 * p_deb) + (0.3 * p_rob)
    
    deb_top1 = np.argmax(p_deb)
    deb_conf = np.max(p_deb)
    deb_top3_str = " ".join([id2label[idx] for idx in np.argsort(p_deb)[-3:][::-1]])
    
    ens_top1 = np.argmax(p_ens)
    ens_conf = np.max(p_ens)
    ens_top3_str = " ".join([id2label[idx] for idx in np.argsort(p_ens)[-3:][::-1]])
    
    # Q7
    if deb_top1 != ens_top1:
        top1_changes += 1
        
    # Q8
    gain = ens_conf - deb_conf
    if gain > 0:
        positive_gain_count += 1
        
    # Q9
    if deb_top3_str != ens_top3_str:
        top3_changes += 1

print(f"Q7 -> {top1_changes}")
print(f"Q8 -> {positive_gain_count}")
print(f"Q9 -> {top3_changes}")

Q7 -> 26
Q8 -> 13
Q9 -> 72


In [17]:
# Q10

def apk(actual, predicted, k=3):
    if not predicted or not actual:
        return 0.0
    if len(predicted) > k:
        predicted = predicted[:k]
    score = 0.0
    num_hits = 0.0
    for i, p in enumerate(predicted):
        if p == actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
    return score

map3_scores = []

for i in range(min(100, len(val_df))):
    text = str(val_df['prompt'].iloc[i])
    true_label = str(val_df['answer'].iloc[i]).strip()
    
    p_deb = get_probs(text, deb_model, deb_tok)
    p_rob = get_probs(text, rob_model, rob_tok)
    p_ens = (0.7 * p_deb) + (0.3 * p_rob)
    
    top3_idx = np.argsort(p_ens)[-3:][::-1]
    top3_letters = [id2label[idx] for idx in top3_idx]
    
    score = apk(true_label, top3_letters, k=3)
    map3_scores.append(score)

final_map3 = np.mean(map3_scores)
print(f"Q10 -> {final_map3:.4f}")

Q10 -> 0.4100


# Submission Cell

In [ ]:
# submission = pd.DataFrame({
#     "ID": test_df["id"],
#     "Prediction": predictions
# })

# submission.to_csv("submission.csv", index=False)

# submission.head()